In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("/Users/biman_giri/Documents/OfficeWork/MyDay2DayLearning/.env")
openai_token = os.getenv("OPENAI_API_TOKEN")
openai_base_url = os.getenv("OPENAI_API_BASE")
lanchain_endpoint = os.getenv("LANGSMITH_ENDPOINT")
lanchain_api_key = os.getenv("LANGSMITH_API_KEY")
lanchain_project = os.getenv("LANGSMITH_PROJECT")
lanchain_tracing_v2 = os.getenv("LANGSMITH_TRACING")
 

In [2]:
from langchain_openai import ChatOpenAI
llm_model = ChatOpenAI(model = "gpt-4o-mini", api_key = openai_token, base_url = openai_base_url)
response = llm_model.invoke("what is the capital of india?")
print(response)


/Users/biman_giri/.pyenv/versions/3.11.4/envs/BimanDS/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


content='The capital of India is New Delhi.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 14, 'total_tokens': 22, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a1ddba3226', 'id': 'chatcmpl-DIMuqrU0bhorNzBfsCN3QqFnMTt9x', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019cdf26-6a77-7f42-8e48-cc69fdc11c7b-0' usage_metadata={'input_tokens': 14, 'output_tokens': 8, 'total_tokens': 22, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [3]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that can answer questions and help with tasks."),
    ("user", "{input}")
])

prompt.invoke({"input": "What is the capital of India?"})


ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant that can answer questions and help with tasks.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of India?', additional_kwargs={}, response_metadata={})])

In [4]:
chain = prompt | llm_model

result = chain.invoke({"input": "What is the capital of India?"})
result.content

'The capital of India is New Delhi.'

In [5]:
from langchain_core.output_parsers import StrOutputParser 

parser = StrOutputParser()
chain = prompt | llm_model | parser

result = chain.invoke({"input": "What is usage of the langsmith"})
print(result)

LangSmith is a tool designed to assist developers and teams who work with language models, particularly in the context of machine learning and natural language processing (NLP). While I don't have specific details about updates or features introduced after October 2023, typical usage scenarios for tools like LangSmith include:

1. **Model Development**: LangSmith can aid in developing AI models that understand and generate human language. This includes fine-tuning language models on specific tasks or datasets.

2. **Testing and Evaluation**: It may provide features for testing model performance, including metrics for accuracy, precision, recall, and F1 score, helping users evaluate how well their models are performing.

3. **Data Management**: LangSmith could help in managing training datasets, including data preprocessing, augmentation, and organization for model training.

4. **Integration**: The tool might help integrate language models into applications, streamlining the process of

In [6]:
from langchain_community.document_loaders import WebBaseLoader
# loadint the wiki page loader
loader = WebBaseLoader("https://en.wikipedia.org/wiki/Men%27s_T20_World_Cup")


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [7]:
t20_world_cup_docs = loader.load()

In [11]:
# lets split the document into chunks

from langchain_text_splitters import CharacterTextSplitter
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    encoding_name = "cl100k_base",
    chunk_size = 1000,
    chunk_overlap = 200
)

# split the document into chunks
t20_world_cup_docs_chunks = text_splitter.split_documents(t20_world_cup_docs)

print(len(t20_world_cup_docs_chunks))




19


In [19]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
openai_embeddings_model = OpenAIEmbeddings(model = "text-embedding-3-small", api_key = openai_token, base_url = openai_base_url)
faiss_db = FAISS.from_documents(
    documents = t20_world_cup_docs_chunks,
    embedding = openai_embeddings_model,
)
faiss_db.save_local("faiss_db")

In [24]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template(
    """
        You are a helpful assistant that can answer questions from the given context.
        Here is the context:
        {context}
        Here is the question:
        {question}
    """
)

In [29]:
while True:
    input_query = input("Enter your question:  ")
    if input_query == "exit":
        break
    related_docs = faiss_db.similarity_search(input_query, k = 3)
    context = "\n".join([doc.page_content for doc in related_docs])
    result = prompt.invoke({"context": context, "question": input_query})
    chain = prompt | llm_model | StrOutputParser()
    result = chain.invoke({"context": context, "question": input_query})
    print(result)



India won the 2024 Men's T20 World Cup, defeating South Africa in the final. This victory marked India's second T20 World Cup title after 17 years.
India won the 2024 T20 World Cup, defeating South Africa in the final to secure their second title after 17 years.
The countries that won the Men's T20 World Cup from 2007 to 2026 are:

1. **2007** - India
2. **2009** - Pakistan
3. **2010** - England
4. **2012** - West Indies
5. **2014** - Sri Lanka
6. **2016** - West Indies
7. **2021** - Australia
8. **2022** - England
9. **2024** - India
10. **2026** - India

In summary, the winning countries are India, Pakistan, England, West Indies, Sri Lanka, and Australia.


### Build a simple LLM application with LCEL
in this quick start we'll show you how to build a simple LLM application with Langchain. This application will trnslate text from English into another language. This is arelatively simple LLM application - it's just a simgle LLM call plus some prompting. Still, this a great way to get started with Langchain - a lot of features can be build with just some prompting and an LLM call!. 

After seeing this video you ' ll have a high level overview of :
- using language models
- using PromptTemplate and OutputParsers
- Using LangChain Expression Language (LCEL) to chain components together.
- Debugging and tracing your application using LangSmith
- deploying your application with LangServe


In [31]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
messages = [
    SystemMessage(content = "Translate the following from English to Bengali?"),
    HumanMessage(content = "I love programming in Python")
]

response = llm_model.invoke(messages)
response.content

'আমি পাইথনে প্রোগ্রামিং করতে ভালোবাসি।'

In [32]:
### print the llm response using the StrOutputParser
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
parser.invoke(response)

'আমি পাইথনে প্রোগ্রামিং করতে ভালোবাসি।'

In [33]:
# Using LCEL to chain the components together
chain = llm_model | parser
chain.invoke(messages)

'আমি পাইথনে প্রোগ্রামিং করতে ভালোবাসি।'

In [37]:
## prompt template

from langchain_core.prompts import ChatPromptTemplate
generate_template = "Translate the following from English to {language}"
prompt = ChatPromptTemplate.from_messages([
    ("system", generate_template),
    ("user", "{text}")
])
result = prompt.invoke({"text": "I love programming in Python", "language": "Bengali"})

result

ChatPromptValue(messages=[SystemMessage(content='Translate the following from English to Bengali', additional_kwargs={}, response_metadata={}), HumanMessage(content='I love programming in Python', additional_kwargs={}, response_metadata={})])

In [36]:
result.to_messages()

[SystemMessage(content='Translate the following from English to Bengali', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I love programming in Python', additional_kwargs={}, response_metadata={})]

In [38]:
chain = prompt | llm_model | parser
chain.invoke({"text": "I love programming in Python", "language": "Bengali"})

'মुझे পাইথন প্রোগ্রামিং করতে ভালো লাগে।'